# 02 — Pré-processamento dos Dados

Este notebook apresenta a preparação dos dados para os problemas de regressão e classificação definidos no projeto.

O pré-processamento inclui a remoção de identificadores, separação dos conjuntos de treino e teste, tratamento de valores ausentes, codificação das variáveis categóricas e padronização das variáveis numéricas.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/ev_battery_failure_dataset.csv")

print(df.shape)

(200000, 70)


## Remoção de identificadores

As variáveis `vehicle_id` e `battery_serial` foram removidas por funcionarem essencialmente como identificadores dos registros. Apesar de permitirem distinguir veículos e baterias, seus valores não representam características diretamente relacionadas ao comportamento físico ou operacional da bateria e, portanto, não foram considerados atributos preditivos.

In [ ]:
# Cópia do dataset para pré-processamento

df_prep = df.copy()

# Remover identificadores que não possuem significado preditivo
df_prep = df_prep.drop(columns=['vehicle_id', 'battery_serial'])

print(f"Dimensões após remoção dos identificadores: {df_prep.shape}")

Dimensões após remoção dos identificadores: (2713, 68)


## Preparação dos conjuntos para cada problema

A partir do mesmo conjunto de dados foram preparados dois conjuntos específicos. Para regressão, o alvo é `predicted_remaining_life_cycles`, enquanto para classificação o alvo é `battery_failure`.

Registros sem valor para o respectivo alvo são removidos antes das etapas seguintes, pois não podem ser utilizados como exemplos supervisionados para aquele problema.

In [ ]:
# Criar os datasets específicos para cada problema

df_reg = df_prep.dropna(subset=['predicted_remaining_life_cycles']).copy()
df_class = df_prep.dropna(subset=['battery_failure']).copy()

print("Dataset de regressão:", df_reg.shape)
print("Dataset de classificação:", df_class.shape)

Dataset de regressão: (2600, 68)
Dataset de classificação: (2712, 68)


In [ ]:
print("Dimensões atuais:", df.shape)
print(df.head())

Dimensões atuais: (200000, 70)
  vehicle_id vehicle_brand vehicle_model vehicle_type  manufacturing_year  \
0   EV100000        Nissan          Leaf          SUV              2019.0   
1   EV100001          Audi        e-tron        Sedan              2018.0   
2   EV100002        Toyota          bZ4X          Van              2018.0   
3   EV100003    Volkswagen          ID.3    Hatchback              2022.0   
4   EV100004         Tesla       Model X    Crossover              2015.0   

  battery_manufacturer battery_chemistry  battery_capacity_kwh drive_type  \
0                 CATL               LMO                 82.54        RWD   
1          Samsung SDI               NaN                 64.17        AWD   
2              Guoxuan               NMC                 90.92        RWD   
3                 CATL               NMC                 47.35        AWD   
4          BYD Battery               NMC                 68.04        FWD   

   odometer_km  ...  sensor_fault_count BMS

In [ ]:
print("Quantidade de veículos:", df['vehicle_id'].nunique())
print("Linhas:", len(df))

Quantidade de veículos: 200000
Linhas: 200000


In [ ]:
df_prep = df.copy()

df_prep = df_prep.drop(
    columns=['vehicle_id', 'battery_serial']
)

print(df_prep.shape)

(200000, 68)


In [ ]:
# Dataset para regressão
df_reg = df_prep.dropna(subset=['predicted_remaining_life_cycles']).copy()

# Dataset para classificação
df_class = df_prep.dropna(subset=['battery_failure']).copy()

print("Dataset de regressão:", df_reg.shape)
print("Dataset de classificação:", df_class.shape)

Dataset de regressão: (191026, 68)
Dataset de classificação: (200000, 68)


## Separação entre treino e teste

Os dados foram divididos em conjuntos de treino (80%) e teste (20%), utilizando `random_state=42` para garantir a reprodutibilidade da divisão.

No problema de classificação foi utilizada a opção `stratify`, preservando aproximadamente a proporção original das classes de `battery_failure` nos conjuntos de treino e teste.

Além disso, o alvo de cada problema foi excluído das características do outro problema, evitando dependência entre os dois alvos definidos no projeto.

In [ ]:
from sklearn.model_selection import train_test_split

# =========================
# REGRESSÃO
# =========================

X_reg = df_reg.drop(columns=['predicted_remaining_life_cycles'])
y_reg = df_reg['predicted_remaining_life_cycles']

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Regressão:")
print("Treino:", X_reg_train.shape)
print("Teste:", X_reg_test.shape)


# =========================
# CLASSIFICAÇÃO
# =========================

X_class = df_class.drop(columns=['battery_failure'])
y_class = df_class['battery_failure']

X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

print("\nClassificação:")
print("Treino:", X_class_train.shape)
print("Teste:", X_class_test.shape)

Regressão:
Treino: (152820, 67)
Teste: (38206, 67)

Classificação:
Treino: (160000, 67)
Teste: (40000, 67)


## Identificação dos tipos de características

As características foram separadas entre numéricas e categóricas, pois cada tipo exige estratégias diferentes de tratamento. Essa separação permite aplicar as transformações adequadas de forma automatizada por meio do `ColumnTransformer`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identificar variáveis numéricas e categóricas
numeric_features = X_reg_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_reg_train.select_dtypes(include='object').columns.tolist()

print("Variáveis numéricas:", len(numeric_features))
print("Variáveis categóricas:", len(categorical_features))
print("\nCategóricas:")
print(categorical_features)

Variáveis numéricas: 59
Variáveis categóricas: 8

Categóricas:
['vehicle_brand', 'vehicle_model', 'vehicle_type', 'battery_manufacturer', 'battery_chemistry', 'drive_type', 'fleet_or_private', 'terrain_type']


## Tratamento e transformação das características

Para as variáveis numéricas, os valores ausentes são preenchidos pela **mediana**, uma medida menos sensível à presença de valores extremos. Em seguida, é aplicado o `StandardScaler`, padronizando as características para média aproximadamente zero e desvio padrão aproximadamente um.

Para as variáveis categóricas, os valores ausentes são preenchidos pela **categoria mais frequente (moda)**. Em seguida, é aplicado **One-Hot Encoding**, pois essas categorias não apresentam uma ordem natural entre seus valores.

Os outliers identificados durante a análise exploratória não foram removidos indiscriminadamente, pois podem representar condições válidas de operação dos veículos e das baterias.

In [ ]:
# Pipeline para variáveis numéricas:
# - valores ausentes -> mediana
# - padronização -> média 0 e desvio padrão 1

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variáveis categóricas:
# - valores ausentes -> moda
# - categorias -> One-Hot Encoding

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinar os dois tratamentos
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Pipeline de pré-processamento criado com sucesso!")

Pipeline de pré-processamento criado com sucesso!


## Aplicação do pré-processamento

O pré-processamento é ajustado exclusivamente utilizando o conjunto de treino e posteriormente aplicado ao conjunto de teste. Essa estratégia evita que informações provenientes do conjunto de teste sejam utilizadas durante a preparação dos dados de treinamento, reduzindo o risco de vazamento de dados (*data leakage*).

In [ ]:
# Ajustar o pré-processamento usando apenas os dados de treino
X_reg_train_prep = preprocessor.fit_transform(X_reg_train)

# Aplicar o mesmo pré-processamento aos dados de teste
X_reg_test_prep = preprocessor.transform(X_reg_test)

print("Regressão:")
print("Treino após pré-processamento:", X_reg_train_prep.shape)
print("Teste após pré-processamento:", X_reg_test_prep.shape)

Regressão:
Treino após pré-processamento: (152820, 164)
Teste após pré-processamento: (38206, 164)


In [ ]:
# Identificar novamente as variáveis numéricas e categóricas
numeric_features_class = X_class_train.select_dtypes(include=np.number).columns.tolist()
categorical_features_class = X_class_train.select_dtypes(include='object').columns.tolist()

print("Variáveis numéricas:", len(numeric_features_class))
print("Variáveis categóricas:", len(categorical_features_class))
print("Categóricas:", categorical_features_class)

Variáveis numéricas: 59
Variáveis categóricas: 8
Categóricas: ['vehicle_brand', 'vehicle_model', 'vehicle_type', 'battery_manufacturer', 'battery_chemistry', 'drive_type', 'fleet_or_private', 'terrain_type']


In [ ]:
preprocessor_class = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_class),
        ('cat', categorical_transformer, categorical_features_class)
    ]
)

# Ajustar somente com o treino
X_class_train_prep = preprocessor_class.fit_transform(X_class_train)

# Transformar o teste
X_class_test_prep = preprocessor_class.transform(X_class_test)

print("Classificação:")
print("Treino após pré-processamento:", X_class_train_prep.shape)
print("Teste após pré-processamento:", X_class_test_prep.shape)

Classificação:
Treino após pré-processamento: (160000, 164)
Teste após pré-processamento: (40000, 164)


## Verificação do pré-processamento

Após as transformações, são realizadas verificações para confirmar a ausência de valores faltantes e o funcionamento da padronização das características numéricas.

In [ ]:
print("Valores ausentes após o pré-processamento:")

print("Regressão - treino:", np.isnan(X_reg_train_prep).sum())
print("Regressão - teste:", np.isnan(X_reg_test_prep).sum())

print("Classificação - treino:", np.isnan(X_class_train_prep).sum())
print("Classificação - teste:", np.isnan(X_class_test_prep).sum())

Valores ausentes após o pré-processamento:
Regressão - treino: 0
Regressão - teste: 0
Classificação - treino: 0
Classificação - teste: 0


In [ ]:
print("Exemplo das variáveis numéricas após a padronização:")
print("Média aproximada:", X_reg_train_prep[:, :len(numeric_features)].mean())
print("Desvio padrão aproximado:", X_reg_train_prep[:, :len(numeric_features)].std())

Exemplo das variáveis numéricas após a padronização:
Média aproximada: 7.965781431340941e-16
Desvio padrão aproximado: 1.0000000000000002


## Conclusão

O pré-processamento resultou em conjuntos de dados preparados para aplicação de algoritmos de aprendizado de máquina. Os valores ausentes foram tratados, as características categóricas foram convertidas para representação numérica e as características numéricas foram padronizadas.

A separação entre treino e teste foi realizada antes do ajuste das transformações, evitando vazamento de informações. Os identificadores foram removidos e os alvos de regressão e classificação foram mantidos separados das características utilizadas em cada problema.

Dessa forma, os dados estão preparados para as análises específicas de regressão e classificação realizadas nos notebooks seguintes.